# Working within a factor of the best algorithm. 

For this part, I will use the data that I have already collected regarding the algo that has the best ERT. This is specifically for the bbob test suite.

This startegy is a smart approach to keeping in mind algorithms that behave similarly. This is method is beneficial for algorithms that have are slightly less peroming than the best algorithm, but that are still worth keeping as their behavoior is very similar to the best algorithm. This contrasts the other method where only the best algo was kept and all other algorithms were discarded. 
## 1) Comparing the best ERT of each dimension, function, target with the algos from all the different years. 

In this part, I will read the bestERT from the file global_best_algos.csv. This bestERT is determinant in the definition of the best algorithms and will determine which algos I can keep. 

We will define a factor (ex: 2), the we will keep all the algorithms that have an ERT that is within 2 times the ERT of the overall bestERT. This is done over evry dimension, function and target.  

The output result of this will give us a CSV file tracking which are these good algorithms for each dimension, function and target precision. 

These are the following steps I did to implement the code:

1) Read the csv file where the best algos appear, and all the information regarding dimension, function and target 
2) Loop over every year. 
3) Create an interactive interface where we ask the user what factor they want. 
4) Inside every year, loop over dimension function and target and check if the ERT is within the factor bounds. 
5) If it is: write the name of the algo, the year, the dimension, the function and the target precsion to a CSV file. 
6) If it isn't then pass and keep going to the next algo.

In [9]:
import pandas as pd
import numpy as np
import cocopp

In [10]:


# ============================================================
# 1. LOAD YOUR CSV WITH BEST ERTs
# ============================================================

df_best = pd.read_csv("results/global_best_algos.csv")

# clean
df_best = df_best[df_best.best_algorithm.notna()]
df_best = df_best[df_best.best_ERT.notna()]

# all known targets (sorted, unique)
all_targets = sorted(df_best["target"].unique())

# dictionary: (dim, func, target) → best_ERT
best_dict = {}
for _, row in df_best.iterrows():
    key = (int(row.dimension), int(row.function_id), float(row.target))
    best_dict[key] = float(row.best_ERT)


# ============================================================
# 2. ASK USER FOR FACTOR
# ============================================================

try:
    factor = float(input("Enter tolerance factor (default=2): ") or 2)
except:
    factor = 2.0

print(f"\nUsing factor = {factor}\n")


# ============================================================
# 3. LOAD COCO DATA BY YEAR
# ============================================================

years = df_best["year"].unique()     # automatically detect existing years
years = sorted(years)

dsl_by_year = {}

for y in years:
    print(f"Loading BBOB data for year {y} ...")
    try:
        dsl_by_year[y] = cocopp.load(f"bbob/{y}/*")
    except:
        print(f"⚠️  Warning: Could not load year {y}. Skipping.")
        continue


# ============================================================
# 4. LOOP USING YOUR EXACT CODE STRUCTURE
# ============================================================

within_rows = []

for year, dsl in dsl_by_year.items():
    
    print(f"\n=== Processing year {year} ===")
    
    dd = dsl.dictByDimFunc()     # same as in your working code

    for dim in sorted(dd.keys()):
        for func in sorted(dd[dim].keys()):
            
            # loop over algorithms
            for ds in dd[dim][func]:
                
                algo = ds.algId

                # loop over targets from your CSV
                for tgt in all_targets:

                    key = (dim, func, tgt)
                    if key not in best_dict:
                        continue     # best ERT not known for this triple

                    best_ert = best_dict[key]

                    # compute ERT for THIS algorithm at THIS target
                    try:
                        ert = float(ds.detERT([tgt])[0])
                    except Exception:
                        continue

                    if not np.isfinite(ert):
                        continue

                    # check whether ERT is within factor
                    if ert <= factor * best_ert:
                        within_rows.append([
                            year,
                            dim,
                            func,
                            tgt,
                            algo,
                            ert,
                            best_ert
                        ])


# ============================================================
# 5. SAVE OUTPUT CSV
# ============================================================

within_df = pd.DataFrame(
    within_rows,
    columns=[
        "year",
        "dimension",
        "function_id",
        "target",
        "algorithm",
        "ERT",
        "best_ERT"
    ]
)
# be careful to change the file path name according o the test suite you are working with
output_path = "results/within_factor_algorithms.csv"
within_df.to_csv(output_path, index=False)

print(f"\n✓ Saved: {output_path}")
print(f"Total rows: {len(within_df)}")


Enter tolerance factor (default=2): 2

Using factor = 2.0

Loading BBOB data for year 2009 ...
Loading BBOB data for year 2010 ...
Loading BBOB data for year 2012 ...
Loading BBOB data for year 2016 ...
Loading BBOB data for year 2017 ...
Loading BBOB data for year 2018 ...
Loading BBOB data for year 2019 ...
Loading BBOB data for year 2020 ...
Loading BBOB data for year 2021 ...
Loading BBOB data for year 2022 ...
Loading BBOB data for year 2023 ...

=== Processing year 2009 ===

=== Processing year 2010 ===

=== Processing year 2012 ===

=== Processing year 2016 ===

=== Processing year 2017 ===

=== Processing year 2018 ===

=== Processing year 2019 ===

=== Processing year 2020 ===

=== Processing year 2021 ===

=== Processing year 2022 ===

=== Processing year 2023 ===

✓ Saved: results/within_factor_algorithms.csv
Total rows: 7760


## 2) Counting the number of times an algo appears for each dimension

In this part, we are going to be aggregating over all the different function and targets within a certain dimension. The goal is to have for each dimension the best performing algorithms with the factor determined above, and also display the count.

In [11]:
import pandas as pd

# ============================================================
# 1. READ THE CSV FROM PART 1
# ============================================================

df = pd.read_csv("results/within_factor_algorithms.csv")

# clean potential NaNs
df = df[df.algorithm.notna()]
df = df[df.ERT.notna()]


# ============================================================
# 2. INITIALIZE THE COUNTER
# ============================================================
# We use a dictionary: (dimension, algorithm) → count

counter = {}

for _, row in df.iterrows():
    dim  = int(row["dimension"])
    algo = row["algorithm"]

    key = (dim, algo)

    if key not in counter:
        counter[key] = 0
    counter[key] += 1


# ============================================================
# 3. CONVERT DICTIONARY TO DATAFRAME
# ============================================================

rows = []
for (dim, algo), count in counter.items():
    rows.append([dim, algo, count])

counts_df = pd.DataFrame(rows, columns=["dimension", "algorithm", "count"])


# Sort: first by dimension, then by descending count
counts_df = counts_df.sort_values(["dimension", "count"], ascending=[True, False])


# ============================================================
# 4. SAVE THE RESULTING CSV
# ============================================================

output_path = "results/counts_by_dimension.csv"
counts_df.to_csv(output_path, index=False)

print(f"\n✓ Saved: {output_path}")
print(f"Total rows: {len(counts_df)}")

# Optional: display a preview
print("\n=== Preview ===")
print(counts_df.head(20))



✓ Saved: results/counts_by_dimension.csv
Total rows: 640

=== Preview ===
     dimension                      algorithm  count
437          2         SLSQP+lq-CMA-ES_Hansen     52
525          2              DIRECT-REV_Kudela     51
5            2              NELDERDOERR_doerr     45
438          2          SLSQP-11-scipy_Hansen     45
439          2               lq-CMA-ES_Hansen     36
281          2               DTS-CMA-ES_Pitra     34
385          2       SLSQP-scipy-2019_Varelas     34
6            2                  NELDER_hansen     33
466          2  SHADE-LM-POP4-to-10_Okulewicz     33
467          2             SHADE-LM_Okulewicz     29
476          2               oMads-Neg_Dahito     29
526          2                  BIRMIN_Kudela     29
148          2                 DE-BFGS_voglis     28
478          2                oMads-2N_Dahito     27
1            2                 FULLNEWUOA_ros     23
4            2                      MCS_huyer     23
154          2          

In [12]:
import pandas as pd

# ============================================================
# 1. LOAD THE COUNTS DATA
# ============================================================

df = pd.read_csv("results/counts_by_dimension.csv")

# Make sure index is clean
df = df.reset_index(drop=True)

# Sort rows by (dimension, descending count)
df = df.sort_values(["dimension", "count"], ascending=[True, False])


# ============================================================
# 2. BUILD DICTIONARY: dim → list of (algo, count)
# ============================================================

ranking_dict = {}

for dim in sorted(df["dimension"].unique()):
    df_dim = df[df["dimension"] == dim]

    # list of tuples: (algorithm, count)
    tuples = list(zip(df_dim["algorithm"], df_dim["count"]))

    ranking_dict[dim] = tuples


# ============================================================
# 3. DETERMINE MAX RANK ACROSS ALL DIMENSIONS
# ============================================================

max_len = max(len(v) for v in ranking_dict.values())  # longest list


# ============================================================
# 4. BUILD THE WIDE RANKING TABLE
# ============================================================

rows = []

for rank in range(max_len):  # 0 = best, 1 = second best, …
    row = {"rank": rank + 1}  # human-readable rank
    
    for dim in sorted(ranking_dict.keys()):
        if rank < len(ranking_dict[dim]):
            algo, count = ranking_dict[dim][rank]
            row[dim] = f"({algo}, {count})"
        else:
            row[dim] = None
    
    rows.append(row)

ranking_table = pd.DataFrame(rows)


# ============================================================
# 5. SAVE TO CSV
# ============================================================

output_path = "results/ranking_table.csv"
ranking_table.to_csv(output_path, index=False)

print(f"\n✓ Saved ranking table to {output_path}\n")


# ============================================================
# 6. DISPLAY CLEAN TABLE
# ============================================================

print("=== Ranking Table (Alg, Count) ===\n")
print(ranking_table)



✓ Saved ranking table to results/ranking_table.csv

=== Ranking Table (Alg, Count) ===

     rank                              2                                    3  \
0       1   (SLSQP+lq-CMA-ES_Hansen, 52)               (lq-CMA-ES_Hansen, 54)   
1       2        (DIRECT-REV_Kudela, 51)         (SLSQP+lq-CMA-ES_Hansen, 41)   
2       3        (NELDERDOERR_doerr, 45)          (SLSQP-11-scipy_Hansen, 37)   
3       4    (SLSQP-11-scipy_Hansen, 45)              (DIRECT-REV_Kudela, 36)   
4       5         (lq-CMA-ES_Hansen, 36)  (SHADE-LM-POP4-to-10_Okulewicz, 34)   
..    ...                            ...                                  ...   
122   123  (Adaptive-Two-Mode_Bodner, 1)                                 None   
123   124            (RS-6_Brockhoff, 1)                                 None   
124   125           (EXPLO2_Huntsman, 1)                                 None   
125   126              (WOA_Espinoza, 1)                                 None   
126   127          (

In [13]:
import pandas as pd

# ============================================================
# 1. LOAD THE COUNTS DATA
# ============================================================

df = pd.read_csv("results/counts_by_dimension.csv")

# Clean and sort
df = df.reset_index(drop=True)
df = df.sort_values(["dimension", "count"], ascending=[True, False])


# ============================================================
# 2. BUILD DICTIONARY: dim → list of (algo, count)
# ============================================================

ranking_dict = {}

for dim in sorted(df["dimension"].unique()):
    df_dim = df[df["dimension"] == dim]
    ranking_dict[dim] = list(zip(df_dim["algorithm"], df_dim["count"]))


# ============================================================
# 3. DETERMINE MAX RANK
# ============================================================

max_len = max(len(v) for v in ranking_dict.values())


# ============================================================
# 4. BUILD WIDE RANKING TABLE
# ============================================================

rows = []

for rank in range(max_len):
    row = {"rank": rank + 1}

    for dim in sorted(ranking_dict.keys()):
        if rank < len(ranking_dict[dim]):
            algo, count = ranking_dict[dim][rank]
            row[f"dim {dim}"] = f"{algo} ({count})"
        else:
            row[f"dim {dim}"] = ""
    
    rows.append(row)

ranking_table = pd.DataFrame(rows)


# ============================================================
# 5. DISPLAY NICELY USING PANDAS STYLING
# ============================================================

styled_table = (
    ranking_table.style
        .set_properties(**{
            "background-color": "#f7f7f7",
            "border": "1px solid #ccc",
            "padding": "6px",
            "font-size": "12px"
        })
        .set_table_styles([
            {"selector": "th", 
             "props": [("background-color", "#e6e6e6"),
                       ("font-weight", "bold"),
                       ("border", "1px solid #aaa"),
                       ("padding", "6px")]}
        ])
        .hide(axis="index")  # Hide the pandas index entirely
)

styled_table


rank,dim 2,dim 3,dim 5,dim 10,dim 20,dim 40
1,SLSQP+lq-CMA-ES_Hansen (52),lq-CMA-ES_Hansen (54),lq-CMA-ES_Hansen (59),SLSQP-11-scipy_Hansen (39),SLSQP+lq-CMA-ES_Hansen (41),SLSQP+lq-CMA-ES_Hansen (43)
2,DIRECT-REV_Kudela (51),SLSQP+lq-CMA-ES_Hansen (41),SLSQP+lq-CMA-ES_Hansen (47),SLSQP+lq-CMA-ES_Hansen (38),IPOPsaACM_loshchilov (36),SLSQP-11-scipy_Hansen (34)
3,NELDERDOERR_doerr (45),SLSQP-11-scipy_Hansen (37),SLSQP-11-scipy_Hansen (46),BIPOPsaACM_loshchilov (35),lq-CMA-ES_Hansen (30),NEWUOA_ros (30)
4,SLSQP-11-scipy_Hansen (45),DIRECT-REV_Kudela (36),SLSQP-scipy-2019_Varelas (36),lq-CMA-ES_Hansen (33),SLSQP-11-scipy_Hansen (29),adm-CMA-ES_Gissler (30)
5,lq-CMA-ES_Hansen (36),SHADE-LM-POP4-to-10_Okulewicz (34),CMA-ES-2019_Hansen (27),IPOPsaACM_loshchilov (27),NIPOPaCMA_loshchilov (27),SLSQP-scipy-2019_Varelas (28)
6,DTS-CMA-ES_Pitra (34),SLSQP-scipy-2019_Varelas (33),SHADE-LM_Okulewicz (26),adm-CMA-ES_Gissler (24),PSA-CMA-ESwRS_Nishida (27),NIPOPaCMA_loshchilov (27)
7,SLSQP-scipy-2019_Varelas (34),DE-BFGS_voglis (32),DE-BFGS_voglis (25),SLSQP-scipy-2019_Varelas (23),BIPOPsaACM_loshchilov (25),BFGS-P-StPt_Blelly (27)
8,NELDER_hansen (33),NELDER_hansen (26),BFGS-P-StPt_Blelly (24),a-CMA-ES_Gissler (23),a-CMA-ES_Gissler (24),a-CMA-ES_Gissler (27)
9,SHADE-LM-POP4-to-10_Okulewicz (33),BFGS-P-StPt_Blelly (26),DIRECT-REV_Kudela (24),HE-ES_Glasmachers (21),BFGS-P-StPt_Blelly (23),ad-CMA-ES_Gissler (26)
10,SHADE-LM_Okulewicz (29),BIRMIN_Kudela (26),a-CMA-ES_Gissler (24),cd-CMA-ES_Gissler (21),SLSQP-scipy-2019_Varelas (23),CMAES-APOP-KMA_Nguyen (24)


In [15]:
# Aggregate counts over all dimensions
overall_counts = (
    df.groupby("algorithm")["count"]
      .sum()
      .reset_index()
      .sort_values("count", ascending=False)
)

print("\nOverall frequency of being best (aggregated over ALL dimensions):\n")
print(overall_counts.head(10))



Overall frequency of being best (aggregated over ALL dimensions):

                    algorithm  count
119    SLSQP+lq-CMA-ES_Hansen    262
139          lq-CMA-ES_Hansen    230
120     SLSQP-11-scipy_Hansen    230
121  SLSQP-scipy-2019_Varelas    177
18         BFGS-P-StPt_Blelly    140
65          DIRECT-REV_Kudela    127
52             DE-BFGS_voglis    123
126          a-CMA-ES_Gissler    122
129        adm-CMA-ES_Gissler    121
97                 NEWUOA_ros    119


In [16]:
overall_counts["percentage"] = (
    overall_counts["count"] / overall_counts["count"].sum() * 100
).round(2)

overall_counts.head(10)


,algorithm,count,percentage
119,SLSQP+lq-CMA-ES_Hansen,262,3.38
139,lq-CMA-ES_Hansen,230,2.96
120,SLSQP-11-scipy_Hansen,230,2.96
121,SLSQP-scipy-2019_Varelas,177,2.28
18,BFGS-P-StPt_Blelly,140,1.80
65,DIRECT-REV_Kudela,127,1.64
52,DE-BFGS_voglis,123,1.59
126,a-CMA-ES_Gissler,122,1.57
129,adm-CMA-ES_Gissler,121,1.56
97,NEWUOA_ros,119,1.53


# 3) Plotting the results using the COCO platform tool

This last step is essential for checking if the result sin our table are indeed coherent. 
Using the COCO tool, we can access make a full analysis of the best performing algorithms. 

## 3.1) Performance of the best algorithms for each dimension

The first comparison between algorithms I want to do, is see the performance of the best algorithms for the differnt dimensionsion.  

In [11]:
cocopp.main(['bbob/2020/SLSQP+lq-CMA-ES_Hansen.tgz','bbob/2020/lq-CMA-ES_Hansen.tgz', 'SLSQP-11-scipy_Hansen',])

Post-processing (2+)
  Using 3 data sets:
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2020\SLSQP+lq-CMA-ES_Hansen.tgz
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2020\lq-CMA-ES_Hansen.tgz
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2020\SLSQP-11-scipy_Hansen.tgz

Post-processing (2+)
  loading data...
  using: C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2020\SLSQP+lq-CMA-ES_Hansen.tgz
  using: C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2020\lq-CMA-ES_Hansen.tgz
  Data consistent according to consistency_check() in pproc.DataSet
  using: C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2020\SLSQP-11-scipy_Hansen.tgz
  Will generate output data in folder ppdata\SLSQP_lq-CM_SLSQP_112016h3028
    this might take several minutes.
ECDF graphs per noise group...
Loading best algorithm data from refalgs/best2009-bbob.tar.gz ...
  using: C:\Users\elsaf\anaconda3\Lib\site-packages\cocopp

DictAlg([(('SLSQP+lq-CMA-ES_Hansen', ''),
          [DataSet(SLSQP+lq-CMA-ES_Hansen on f1 2-D),
           DataSet(SLSQP+lq-CMA-ES_Hansen on f2 2-D),
           DataSet(SLSQP+lq-CMA-ES_Hansen on f3 2-D),
           DataSet(SLSQP+lq-CMA-ES_Hansen on f4 2-D),
           DataSet(SLSQP+lq-CMA-ES_Hansen on f5 2-D),
           DataSet(SLSQP+lq-CMA-ES_Hansen on f6 2-D),
           DataSet(SLSQP+lq-CMA-ES_Hansen on f7 2-D),
           DataSet(SLSQP+lq-CMA-ES_Hansen on f8 2-D),
           DataSet(SLSQP+lq-CMA-ES_Hansen on f9 2-D),
           DataSet(SLSQP+lq-CMA-ES_Hansen on f10 2-D),
           DataSet(SLSQP+lq-CMA-ES_Hansen on f11 2-D),
           DataSet(SLSQP+lq-CMA-ES_Hansen on f12 2-D),
           DataSet(SLSQP+lq-CMA-ES_Hansen on f13 2-D),
           DataSet(SLSQP+lq-CMA-ES_Hansen on f14 2-D),
           DataSet(SLSQP+lq-CMA-ES_Hansen on f15 2-D),
           DataSet(SLSQP+lq-CMA-ES_Hansen on f16 2-D),
           DataSet(SLSQP+lq-CMA-ES_Hansen on f17 2-D),
           DataSet(SLSQP+lq-CMA-

## 3.2) Comparison of the 3 best algorithms for the dimension 2


In [3]:
cocopp.main(['bbob/2020/SLSQP+lq-CMA-ES_Hansen.tgz','DIRECT-REV_Kudela', 'NELDERDOERR_doerr',])

Post-processing (2+)
  Using 3 data sets:
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2020\SLSQP+lq-CMA-ES_Hansen.tgz
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2023\DIRECT-REV_Kudela.tgz
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2009\NELDERDOERR_doerr_noiseless.tgz

Post-processing (2+)
  loading data...
  using: C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2020\SLSQP+lq-CMA-ES_Hansen.tgz
  using: C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2023\DIRECT-REV_Kudela.tgz
  using: C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2009\NELDERDOERR_doerr_noiseless.tgz
  Will generate output data in folder ppdata\SLSQP_DIREC_NELDE_112416h0626
    this might take several minutes.
ECDF graphs per noise group...
Loading best algorithm data from refalgs/best2009-bbob.tar.gz ...
  using: C:\Users\elsaf\anaconda3\Lib\site-packages\cocopp\refalgs/best2009-bbob.tar.gz
  done (Mon Nov 24 16:06

DictAlg([(('SLSQP+lq-CMA-ES_Hansen', ''),
          [DataSet(SLSQP+lq-CMA-ES_Hansen on f1 2-D),
           DataSet(SLSQP+lq-CMA-ES_Hansen on f2 2-D),
           DataSet(SLSQP+lq-CMA-ES_Hansen on f3 2-D),
           DataSet(SLSQP+lq-CMA-ES_Hansen on f4 2-D),
           DataSet(SLSQP+lq-CMA-ES_Hansen on f5 2-D),
           DataSet(SLSQP+lq-CMA-ES_Hansen on f6 2-D),
           DataSet(SLSQP+lq-CMA-ES_Hansen on f7 2-D),
           DataSet(SLSQP+lq-CMA-ES_Hansen on f8 2-D),
           DataSet(SLSQP+lq-CMA-ES_Hansen on f9 2-D),
           DataSet(SLSQP+lq-CMA-ES_Hansen on f10 2-D),
           DataSet(SLSQP+lq-CMA-ES_Hansen on f11 2-D),
           DataSet(SLSQP+lq-CMA-ES_Hansen on f12 2-D),
           DataSet(SLSQP+lq-CMA-ES_Hansen on f13 2-D),
           DataSet(SLSQP+lq-CMA-ES_Hansen on f14 2-D),
           DataSet(SLSQP+lq-CMA-ES_Hansen on f15 2-D),
           DataSet(SLSQP+lq-CMA-ES_Hansen on f16 2-D),
           DataSet(SLSQP+lq-CMA-ES_Hansen on f17 2-D),
           DataSet(SLSQP+lq-CMA-

In [ ]:
cocopp.main(['bbob/2020/SLSQP+lq-CMA-ES_Hansen.tgz','SLSQP-11-scipy_Hansen', 'NEWUOA_ros'])    